# Introducción al Análisis de Datos con Python


# 5. Remodelación de datos


---

En este notebook, realizaremos una serie de procesos de transformación y remodelación de datos que nos facilitaran el análisis y visualización. Estos procesos consisten en unir subconjuntos de datos y generar tablas derivadas para analizar nuestros datos y como datos de entrada para pruebas estadísticas.

Trabajaremos con las Estadísticas de Defunciones Registradas (EDR) de 2020, una base de datos generada por el Instituto Nacional de Estadística y Geografía (INEGI) de México.

Este ejercicio te servirá como base para el proyecto final del curso, donde aplicarás estas mismas etapas y técnicas en un subconjunto de datos para resolver un problema de tu interés.

Las opciones de transformación y remodelación de datos a revisar son:
* **Tablas pivote**: tablas resumen para analizar datos con múltiples variables categóricas con `pd.pivot_table()`.
* **Combinación de DataFrames**: unir información de múltiples tablas usando `pd.merge()` y `pd.concat()`.
* **Agregación de datos**: calcular estadísticas resumidas (como el conteo de muertes, promedios de edad, etc.) para diferentes grupos de interés con `.groupby()`.

En primer lugar, es necesario cargar el conjunto de datos **limpio** que trabajamos en la sección anterior.

In [70]:
# Instalar e importar bibliotecas
import pandas as pd

# Abrir datos
from google.colab import drive
drive.mount('/content/drive')

file_name = '/content/drive/MyDrive/Trabajo/Python_EDR2020/defunciones_covid19_2020.parquet'
df = pd.read_parquet(file_name)
df

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,fecha_ocur,dia_ocurr,mes_ocurr,anio_ocur,ent_ocurr,ent_ocurr_nom,mun_ocurr,mun_ocurr_nom,causa_def,causa_def_nom,sexo,edad,edad_agru
0,2020-10-26,26.0,10.0,2020,1,Aguascalientes,1,Aguascalientes,U071,"COVID-19, virus identificado",Hombre,58.0,De 55 a 59
1,2020-10-26,26.0,10.0,2020,1,Aguascalientes,1,Aguascalientes,U071,"COVID-19, virus identificado",Hombre,72.0,De 70 a 74
2,2020-04-28,28.0,4.0,2020,1,Aguascalientes,1,Aguascalientes,U071,"COVID-19, virus identificado",Hombre,64.0,De 60 a 64
3,2020-05-13,13.0,5.0,2020,1,Aguascalientes,1,Aguascalientes,U071,"COVID-19, virus identificado",Hombre,46.0,De 45 a 49
4,2020-05-17,17.0,5.0,2020,1,Aguascalientes,1,Aguascalientes,U071,"COVID-19, virus identificado",Hombre,45.0,De 45 a 49
...,...,...,...,...,...,...,...,...,...,...,...,...,...
200258,2020-12-31,31.0,12.0,2020,32,Zacatecas,56,Zacatecas,U071,"COVID-19, virus identificado",Mujer,57.0,De 55 a 59
200259,2020-12-31,31.0,12.0,2020,32,Zacatecas,56,Zacatecas,U072,"COVID-19, virus no identificado",Mujer,68.0,De 65 a 69
200260,2020-12-11,11.0,12.0,2020,32,Zacatecas,17,Guadalupe,U071,"COVID-19, virus identificado",Hombre,74.0,De 70 a 74
200261,2020-12-18,18.0,12.0,2020,32,Zacatecas,17,Guadalupe,U071,"COVID-19, virus identificado",Mujer,81.0,De 80 a 84


Revisemos la información del DataFrame. Nota como las columnas de tipo `category` se guardaron como `object`. Esta es una limitación del formato `parquet`, sin embargo, en este momento podemos trabajar con ellas así.

In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200263 entries, 0 to 200262
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   fecha_ocur     200233 non-null  datetime64[ns]
 1   dia_ocurr      200233 non-null  float64       
 2   mes_ocurr      200240 non-null  float64       
 3   anio_ocur      200263 non-null  int64         
 4   ent_ocurr      200263 non-null  int64         
 5   ent_ocurr_nom  200263 non-null  object        
 6   mun_ocurr      200263 non-null  int64         
 7   mun_ocurr_nom  200263 non-null  object        
 8   causa_def      200263 non-null  object        
 9   causa_def_nom  200263 non-null  object        
 10  sexo           200263 non-null  category      
 11  edad           200195 non-null  float64       
 12  edad_agru      200263 non-null  category      
dtypes: category(2), datetime64[ns](1), float64(3), int64(3), object(4)
memory usage: 17.2+ MB


## `pivot_table`: relacionar variables

Una `tabla pivote` es una tabla dinámica que permite reorganizar y resumir datos de forma que faciliten su análisis al relacionar dos o más variables. Son útiles para calcular estadísticas resumidas como sumas, medias o conteos agrupados por diferentes categorías.
Pare realizar una tabla pivote es necesario que el conjunto de datos se encuentre **ordenado** o en formato tidy data: cada variable forma una columna y cada observación forma una fila.
Esta función se encuentra disponible en muchas aplicaciones de hoja de cálculo y lenguajes de programación, en `pandas` usaremos la función `pd.pivot_table()`.

La estructura típica de una tabla pivote se compone de las siguientes partes:
* **Filas (índice)**: las categorías principales por las que se agrupa el conjunto de datos.
* **Columnas**: las variables o categorías adicionales a relacionar y por las cuales se distribuyen los datos.
* **Valores**: los datos que se resumen en la intersección entre filas y columnas. Estos valores pueden ser sumatorias, promedios, conteos o cualquier otra función de agregación aplicada a los datos originales.
* **Campos adicionales** (opcional): algunas tablas pivote incluyen filtros adicionales para permitir una segmentación dinámica de los datos, como seleccionar solo un rango de fechas o una categoría específica.

Por ejemplo, supongamos que se quiere ver la relación entre el sexo y el mes de defunción. Lo primero que se debe de hacer es imaginar cómo se verá la tabla, colocando las filas, columnas y valores.


|       | Hombre | Mujer |
|-------|-----|-----|
| Marzo | ### | ### |
| Abril | ### | ### |
| Mayo  | ### | ### |
| ...   | ### | ### |

donde '###'  representa es el número de ingresos de ese tipo de restos en el año

A continuación es necesario determinar donde está la información en el conjunto de datos ordenado o si no existe calcularla.

* Filas:  `mes_ocurr`
* Columnas: `sexo`
* Función: función `size()`

Basándonos en esto podemos escribir la función correspondiente.

**Nota**: La función `count()` es muy similar a `size()`, pero se aplica a cada columna y no cuenta los `nan`, por lo que los valores pueden variar.

In [72]:
# tabla pivote
pd.pivot_table(df,
               index  = 'mes_ocurr',
               columns= 'sexo',
               aggfunc= 'size'
               )

/tmp/ipykernel_1810/2649903085.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pd.pivot_table(df,


sexo,Hombre,Mujer,No especificado
mes_ocurr,,,
3.0,77,25,0
4.0,4099,1873,0
5.0,14939,7232,1
6.0,17679,9097,0
7.0,19738,11114,1
8.0,15608,8608,1
9.0,10163,6026,0
10.0,10467,6176,1
11.0,13963,8314,1


Es posible generar tablas pivote mucho más complicadas a partir de un conjunto de datos ordenados.

Existen una gran cantidad de funciones que se pueden usar, algunas funciones útiles son:
* `mean`: promedio
* `median`: mediana
* `std`: desviación estándar
* `sum`: suma de valores
* `size`: tamaño del grupo
* `count`: conteo del grupo
* `first`: primer valor
* `last`: último valor
* `nth`: n-esimo valor
* `min`: valor mímino
* `max`: valor máximo

Es recomendable revisar la documentación de [funciones de agrupamiento](https://pandas.pydata.org/pandas-docs/stable/reference/groupby.html#seriesgroupby-computations-descriptive-stats). Además, se pueden usar funciones de otras bibliotecas como `numpy` o definir funciones especiales.

Por ejemplo, se puede calcular la edad promedio de defunción para cada uno de los grupos anteriores. En este caso necesitamos definir sobre que variable y con que función calcularemos el valor de las celdas de la tabla pivote.

* Filas:  `mes_ocurr`
* Columna: `sexo`
* Valores: `edad`
* Función: `mean()`

In [73]:
# tabla pivote
pd.pivot_table(df,
               index  = 'mes_ocurr',
               columns= 'sexo',
               values  = 'edad', # <- variable sobre la que se a a calcular
               aggfunc = 'mean', # <- la operación
               )

/tmp/ipykernel_1810/1176669654.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pd.pivot_table(df,


sexo,Hombre,Mujer,No especificado
mes_ocurr,,,
3.0,57.613333,64.080000,NaN
4.0,57.598975,60.902830,NaN
5.0,59.522409,62.139815,0.0
6.0,61.213028,63.322964,NaN
7.0,62.375881,64.014486,NaN
8.0,63.027813,64.761097,NaN
9.0,63.594685,65.342516,NaN
10.0,63.696637,65.622348,76.0
11.0,64.358269,65.789031,NaN


Supongamos que nos interesa saber la relación entre el mes, el sexo y si es menor de edad, adulto o adulto mayor (60+ años). La tabla se vería:


|      | Hombre menor | Hombre adulto | Hombre mayor | Mujer menor | Mujer adulta | Mujer mayor |
|-------|-----|-----|-----|-----|-----|-----|
| Marzo | ### | ### |-----|-----|-----|-----|
| Abril | ### | ### |-----|-----|-----|-----|
| Mayo  | ### | ### |-----|-----|-----|-----|
| ...   | ### | ### |-----|-----|-----|-----|


A continuación es necesario determinar donde está la información en el conjunto de datos ordenado.

* Filas: `sexo` y `edad` (calcular)
* Columnas: `mes_ocurr`
* Función: `size`

Si no existe directamente se debe calcular, en ese caso se toma la columna `edad` y se define una función especial para dividir en los grupos de edad.

Se pueden usar varios variables o columnas para generar la tabla pivote. Esto genera un [multi-índice](https://pandas.pydata.org/docs/user_guide/advanced.html#). Podemos volver el multi-índice en columnas con la función `.reset_index()`.

In [74]:
# Función para clásificar por edad
def clasificar_edad(edad):
    if   edad <  18 : return "Menor"
    elif edad <  60 : return "Adulto"
    elif edad >= 60 : return "Mayor"

# Aplicar la función y guardar en nueva columna
df['edad_class'] = df['edad'].apply(clasificar_edad)
display( df['edad_class'].value_counts() )

# Tabla pivote
pd.pivot_table(df,
               index = 'mes_ocurr',
               columns = ['sexo','edad_class'],  # <- multiples variables
               aggfunc = 'size'
              )

,count
edad_class,
Mayor,124254
Adulto,75349
Menor,592


/tmp/ipykernel_1810/1145342824.py:12: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pd.pivot_table(df,


sexo       Hombre               Mujer             No especificado            
edad_class Adulto  Mayor Menor Adulto Mayor Menor          Adulto Mayor Menor
mes_ocurr                                                                    
3.0            43     31     1     11    14     0               0     0     0
4.0          2296   1793     8    812  1052     9               0     0     0
5.0          7359   7533    35   2949  4253    29               0     0     1
6.0          7728   9891    50   3345  5715    37               0     0     0
7.0          7904  11773    50   3843  7220    51               0     0     0
8.0          5942   9622    40   2778  5802    26               0     0     0
9.0          3654   6467    39   1818  4181    27               0     0     0
10.0         3810   6623    33   1836  4310    29               0     1     0
11.0         4762   9163    31   2511  5782    21               0     0     0
12.0         8014  13994    40   3924  9022    35               0     0     0

## Cálculo de totales y porcentajes

Es posible usar una tabla como base de otras operaciones, lo cual vuelve está una estructura muy poderosa. Por ejemplo, se pueden calcular los totales, porcentajes y proporciones  de filas y/o columnas. Además, muchas pruebas estadísticas requieren una tabla de contingencia, la cual puede ser calculada como una tabla pivote.

Por ejemplo, usemos la tabla pivote que relaciona las defunciones por sexo y mes de ocurrencia para calcular la suma total de filas y columnas. En este caso usaremos la función `.sum()` para calcular la suma, nota como el parámetro `axis` determina si se suma por filas (`axis=1`) o por columnas (`axis=0`).

Adicionalmente, usaremos `.rename()` para fijar el nombre de la columna.

In [75]:
# tabla pivote a trabajar
data = pd.pivot_table(df, index='mes_ocurr', columns='sexo', aggfunc='size')
data

/tmp/ipykernel_1810/4261006728.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  data = pd.pivot_table(df, index='mes_ocurr', columns='sexo', aggfunc='size')


sexo,Hombre,Mujer,No especificado
mes_ocurr,,,
3.0,77,25,0
4.0,4099,1873,0
5.0,14939,7232,1
6.0,17679,9097,0
7.0,19738,11114,1
8.0,15608,8608,1
9.0,10163,6026,0
10.0,10467,6176,1
11.0,13963,8314,1


In [76]:
# suma de filas
total_filas = data.sum(axis=1).rename('Total filas')
total_filas

,Total filas
mes_ocurr,
3.0,102
4.0,5972
5.0,22172
6.0,26776
7.0,30853
8.0,24217
9.0,16189
10.0,16644
11.0,22278


In [77]:
# suma de columnas
total_columnas = data.sum(axis=0).rename('Total columnas')
total_columnas

,Total columnas
sexo,
Hombre,128789
Mujer,71446
No especificado,5


Podemos obtener el porcentaje de defunciones de las filas y columnas dividiendo por la suma de la tabla y multiplicando por cien.

> Nota: fijate en la sintaxis, rodeamos las operaciones con un parentesis y luefo usamos la función `.rename()` sobre el resultado de todo lo que esta dentro del parentesis.

In [78]:
# Porcentaje filas
porcen_filas = (total_filas * 100 / total_filas.sum()).rename('Porcentaje filas')
porcen_filas

,Porcentaje filas
mes_ocurr,
3.0,0.050939
4.0,2.982421
5.0,11.072713
6.0,13.371954
7.0,15.408010
8.0,12.093987
9.0,8.084798
10.0,8.312026
11.0,11.125649


In [79]:
# Porcentaje columnas
porcen_columnas = (total_columnas * 100 / total_columnas.sum()).rename('Porcentaje columnas')
porcen_columnas

,Porcentaje columnas
sexo,
Hombre,64.317319
Mujer,35.680184
No especificado,0.002497


## `concat`: Unir tablas uno a uno

Es posible unir tablas a lo largo de un eje usando pandas con el comando `pd.concat()`. Esta función toma una lista de DataFrames y las une por el índice o las columnas. En este caso es una unión uno a uno, es decir, cada valor del índice o columna es único y no se repiten.

Por ejemplo, en las sección anterior calculamos las tablas:
* `data`: tabla pivote de decesos por sexo y mes de ocurrencia
* `total_filas`: total de decesos por mes de ocurrencia
* `porcentaje_filas`: porcentaje de decesos por mes de ocurrencia

Supongamos que queremos unir las tres tablas, para poder ver toda la información en una sola tabla. Haremos esto con `pd.concat()`. Esta función reibe una lista de `DataFrames` (o `Series`) y las "pega" en una sola tabla.
Por defecto, esta función concatena las tablas utilizando los indices. El parámetro `axis` determina si se concatenan una sobre la otra (`axis=0`) o lado a lado (`axis=1`).


In [80]:
# Concatenar tablas lado a lado usando el índice
pd.concat([data, total_filas, porcen_filas], axis=1)

,Hombre,Mujer,No especificado,Total filas,Porcentaje filas
mes_ocurr,,,,,
3.0,77,25,0,102,0.050939
4.0,4099,1873,0,5972,2.982421
5.0,14939,7232,1,22172,11.072713
6.0,17679,9097,0,26776,13.371954
7.0,19738,11114,1,30853,15.408010
8.0,15608,8608,1,24217,12.093987
9.0,10163,6026,0,16189,8.084798
10.0,10467,6176,1,16644,8.312026
11.0,13963,8314,1,22278,11.125649


Adicionalmente calculamos dos tablas sobre las columnas:

* `total_columnas`: total de decesos por sexo
* `porcentaje_columnas`: porcentaje de decesos por sexo

Si queremos unirlas sería necesario ponerlas "abajo" de la tabla original cómo filas nuevas. Para lograr esto lo primero es convertirlas de `Series` a `DataFrame` con `.to_frame` y rotarlas con la función `.transpose()`

In [81]:
total_columnas = total_columnas.to_frame().transpose()
display(total_columnas)
porcen_columnas = porcen_columnas.to_frame().transpose()
display(porcen_columnas)

sexo,Hombre,Mujer,No especificado
Total columnas,128789,71446,5


sexo,Hombre,Mujer,No especificado
Porcentaje columnas,64.317319,35.680184,0.002497


Ahora podemos unirlas con `.concat()`, en este caso como queremos ponerlas una sobre la otra usamos `axis=0`. La función usará los nombres de columna cómo guía.

In [82]:
pd.concat([data, total_columnas, porcen_columnas], axis=0)

sexo,Hombre,Mujer,No especificado
3.0,77.000000,25.000000,0.000000
4.0,4099.000000,1873.000000,0.000000
5.0,14939.000000,7232.000000,1.000000
6.0,17679.000000,9097.000000,0.000000
7.0,19738.000000,11114.000000,1.000000
8.0,15608.000000,8608.000000,1.000000
9.0,10163.000000,6026.000000,0.000000
10.0,10467.000000,6176.000000,1.000000
11.0,13963.000000,8314.000000,1.000000
12.0,22056.000000,12981.000000,0.000000


Es posible realizar varias concatenaciones juntas si se guardan los resultados en una tabla y se va modificando repetidamente. Nota como las celdas de la esquina inferior izquierda quedan vacías, ya que ninguna de nuestras tablas incluye esa combinación de filas y columnas.

In [83]:
# Unir total y porcentaje filas
data_total = pd.concat([data, total_filas, porcen_filas], axis=1)
# Unir total y porcentaje columnas
data_total = pd.concat([data_total, total_columnas, porcen_columnas], axis=0)
# Mostrar
data_total

,Hombre,Mujer,No especificado,Total filas,Porcentaje filas
3.0,77.000000,25.000000,0.000000,102.0,0.050939
4.0,4099.000000,1873.000000,0.000000,5972.0,2.982421
5.0,14939.000000,7232.000000,1.000000,22172.0,11.072713
6.0,17679.000000,9097.000000,0.000000,26776.0,13.371954
7.0,19738.000000,11114.000000,1.000000,30853.0,15.408010
8.0,15608.000000,8608.000000,1.000000,24217.0,12.093987
9.0,10163.000000,6026.000000,0.000000,16189.0,8.084798
10.0,10467.000000,6176.000000,1.000000,16644.0,8.312026
11.0,13963.000000,8314.000000,1.000000,22278.0,11.125649
12.0,22056.000000,12981.000000,0.000000,35037.0,17.497503


## `merge`: Unir tablas muchos a uno

Otro caso de unir tablas es cuando se tiene una relación de **muchos a uno**. Esto ocurre cuando múltiples filas en una tabla se corresponden con una única fila en otra tabla. En este contexto, unimos un conjunto de datos primario con un catálogo o una tabla de referencia.

Por ejemplo, recordemos el mapeo del catálogo de entidades y municipios. Originalmente, el EDR contiene los códigos correspondientes a los diferentes niveles geográficos en las variables `ent_ocurr` y `ent_ocurr`. Calculamos las defunciones por estado y municipio y la guardaremos en `data`. Nota cómo la entidad aparece múltiples veces.



In [84]:
data = df[['ent_ocurr', 'mun_ocurr']].value_counts().reset_index()
data

,ent_ocurr,mun_ocurr,count
0,14,39,5445
1,9,7,4946
2,21,114,4842
3,9,5,4812
4,19,39,4338
...,...,...,...
1468,31,61,1
1469,31,63,1
1470,31,71,1
1471,31,74,1


Por otro lado, tenemos el catálogo qué contiene las claves geográficas y nombres del Marco Geoestadístico del INEGI.

In [85]:
# Cargar marco geoestadístico
file_path = '/content/drive/MyDrive/Trabajo/Python_EDR2020/conjunto_de_datos_defunciones_registradas_2020_csv/catalogos/entidad_municipio_localidad_2020.CSV'
mapeo_lugar = pd.read_csv(file_path)
# Seleccionar solo entidades para generar tabla mapeo
mapeo_ent = mapeo_lugar.loc[mapeo_lugar['cve_mun']==0, ['cve_ent','nom_loc']]
display(mapeo_ent.head(10))

,cve_ent,nom_loc
0,1,Aguascalientes
189,2,Baja California
372,3,Baja California Sur
443,4,Campeche
662,5,Coahuila de Zaragoza
973,6,Colima
1089,7,Chiapas
3453,8,Chihuahua
4059,9,Ciudad de México
4131,10,Durango


Ahora, ambas tablas (`data` y `mapeo_ent`) comparten una columna con la clave de la entidad (`ent_ocurr` en `data` y `cve_ent` en `mapeo_ent`), lo que permite unirlas en una sola. Para esto usamos la función `pd.merge()`.

Los parámetros principales de `pd.merge()` son:

  * **`left` y `right`**: Son los dos DataFrames que se van a unir a la izquiera y derecha.
  * **`on`**: La(s) columna(s) común(es) que se usa(n) para unir las tablas. Si las columnas tienen nombres diferentes en cada DataFrame, se usan los parámetros `left_on` y `right_on`.
  * **`how`**: Define el tipo de unión, similar a las operaciones de SQL.

El parámetro `how` acepta las siguientes opciones:

  * **`inner` (unión interna)**: El valor por defecto. Devuelve solo las filas que tienen un valor coincidente en la columna de unión en **ambos** DataFrames. Es el resultado de la intersección.
  * **`outer` (unión externa)**: Devuelve todas las filas de **ambos** DataFrames. Donde no hay coincidencias, se rellenan los valores con `NaN`.
  * **`left` (unión izquierda)**: Devuelve todas las filas del DataFrame `left` y las filas coincidentes del DataFrame `right`. Donde no hay coincidencia en la derecha, se llenan con `NaN`.
  * **`right` (unión derecha)**: Devuelve todas las filas del DataFrame `right` y las filas coincidentes del DataFrame `left`. Donde no hay coincidencia en la izquierda, se llenan con `NaN`.

Para nuestro ejemplo, usaremos la opción `how='left'`, ya que queremos conservar todos los registros de defunciones (`data`) y añadirles los nombres de las entidades correspondientes del catálogo (`mapeo_ent`).

In [86]:
data_nom = pd.merge(
    left=data,
    right=mapeo_ent,
    left_on='ent_ocurr',
    right_on='cve_ent',
    how='left'
)
data_nom

,ent_ocurr,mun_ocurr,count,cve_ent,nom_loc
0,14,39,5445,14,Jalisco
1,9,7,4946,9,Ciudad de México
2,21,114,4842,21,Puebla
3,9,5,4812,9,Ciudad de México
4,19,39,4338,19,Nuevo León
...,...,...,...,...,...
1468,31,61,1,31,Yucatán
1469,31,63,1,31,Yucatán
1470,31,71,1,31,Yucatán
1471,31,74,1,31,Yucatán


Se pueden unir tablas usando múltiples columnas, en ese caso solo se unirán si los valores en ambas columnas coinciden. Al unir las tablas, las columnas que no son parte de la clave de unión se incluyen en el resultado. Si hay columnas con el mismo nombre en ambos DataFrames (y no son la clave de unión), `pandas` añade automáticamente un sufijo (`_x` y `_y`) para evitar conflictos. Puedes personalizar estos sufijos con el parámetro `suffixes`.

Por ejemplo, vamos a unir los nombres de los municipios. Por la forma en la que está conformado el catálogo estos se encuentran en la misma columna que los nombres de entidad `nom_loc`.

In [87]:
# Seleccionar solo las entidades y columnas de interes
mapeo_mun = mapeo_lugar.loc[mapeo_lugar['cve_loc']==0, ['cve_ent', 'cve_mun','nom_loc']]
mapeo_mun

,cve_ent,cve_mun,nom_loc
0,1,0,Aguascalientes
1,1,1,Aguascalientes
27,1,2,Asientos
57,1,3,Calvillo
81,1,4,Cosío
...,...,...,...
30636,35,999,Municipio no especificado
30638,88,0,Entidad no aplica para A00 - R99 Y V90 - Y89
30639,88,888,Municipio no aplica para A00 - R99 Y V90 - Y89
30641,99,0,Entidad no especificada


In [88]:
data_nom = pd.merge(
    left=data_nom,
    right=mapeo_mun,
    left_on=['ent_ocurr','mun_ocurr'],
    right_on=['cve_ent','cve_mun'],
    how='left'
)
data_nom

,ent_ocurr,mun_ocurr,count,cve_ent_x,nom_loc_x,cve_ent_y,cve_mun,nom_loc_y
0,14,39,5445,14,Jalisco,14,39,Guadalajara
1,9,7,4946,9,Ciudad de México,9,7,Iztapalapa
2,21,114,4842,21,Puebla,21,114,Puebla
3,9,5,4812,9,Ciudad de México,9,5,Gustavo A. Madero
4,19,39,4338,19,Nuevo León,19,39,Monterrey
...,...,...,...,...,...,...,...,...
1468,31,61,1,31,Yucatán,31,61,Río Lagartos
1469,31,63,1,31,Yucatán,31,63,Samahil
1470,31,71,1,31,Yucatán,31,71,Sudzal
1471,31,74,1,31,Yucatán,31,74,Tahmek


## División-aplicación-combinación

El pipeline [split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html) es un enfoque muy común en el análisis de datos y se refiere a un proceso de tres pasos:
* _Split_ (divide): El conjunto de datos original se divide en grupos basados en una o más columnas. En primer lugar, se divide el conjunto de datos en grupos basados en una o varias columnas utilizando la función `groupby()`. Esto genera un objeto `GroupBy` que contiene los grupos y se puede aplicar una función o transformación a cada grupo por separado.
* _Apply_ (aplicación): Se aplica alguna función o transformación a cada grupo por separado. Luego, se aplica alguna función o transformación a cada grupo por separado. Esta función puede ser de varios tipos:
    * _Aggregate_ (agregar): calcular una estadística de resumen (o varias estadísticas) para cada grupo. Por ejemplo, calcular la suma o tamaño de los grupos.
    * _Transform_ (transformar): realizar algunos cálculos específicos del grupo y devolver un objeto con el mismo índice.  Por ejemplo, rellenar los valores faltantes (`nan`) dentro de un grupo con un valor derivado de ese mismo grupo.
    * _Filter_ (filtrar): descartar algunos grupos, según un cálculo por grupo que evalúa Verdadero o Falso. Por ejemplo, descartar datos que pertenecen a grupos con muy pocos miembros.
    * Una combinación de lo anterior
* _Combine_ (combinar): Los resultados de cada grupo se combinan en un único DataFrame utilizando la función `concat()`, `merge()` u otra función de combinación de pandas.

A continuación veremos una introducción a este pipeline.

La función `groupby()` se utiliza para agrupar datos en función de una o varias columnas y aplicar una función de agregación a cada grupo.
Esta función divide la tabla o DataFrame en sub-tablas o grupos de acuerdo a los valores de una o más columnas. El resultado es un objeto similar a un diccionario, donde las llaves son los valores por los que se dividió la tabla y los valores las sub-tablas resultantes. Este objeto es iterable y se le pueden aplicar funciones por separado a cada grupo.


In [89]:
# Agrupar por columna
groups = df.groupby('sexo', observed=True)
# Iterar por cada grupo
for key, data in groups:
    print(f"Grupo:{key}, \tTamaño:{data.shape} \tType':{type(data)}")

Grupo:Hombre, 	Tamaño:(128799, 14) 	Type':<class 'pandas.core.frame.DataFrame'>
Grupo:Mujer, 	Tamaño:(71459, 14) 	Type':<class 'pandas.core.frame.DataFrame'>
Grupo:No especificado, 	Tamaño:(5, 14) 	Type':<class 'pandas.core.frame.DataFrame'>


Usando varias columnas para agrupar se genera un `multiindex`.

Esto crea una agrupación jerárquica que considera las combinaciones posibles entre `sexo` y `mes_ocurr`.

> Nota: Para incluir los `nan` en los grupos se usa la opción `dropna=False`

In [90]:
groups = df.groupby(['sexo','mes_ocurr'])

for key, data in groups:
    print(f"Grupo:{key}, \tTamaño:{data.shape} \tType':{type(data)}")

Grupo:('Hombre', np.float64(3.0)), 	Tamaño:(77, 14) 	Type':<class 'pandas.core.frame.DataFrame'>
Grupo:('Hombre', np.float64(4.0)), 	Tamaño:(4099, 14) 	Type':<class 'pandas.core.frame.DataFrame'>
Grupo:('Hombre', np.float64(5.0)), 	Tamaño:(14939, 14) 	Type':<class 'pandas.core.frame.DataFrame'>
Grupo:('Hombre', np.float64(6.0)), 	Tamaño:(17679, 14) 	Type':<class 'pandas.core.frame.DataFrame'>
Grupo:('Hombre', np.float64(7.0)), 	Tamaño:(19738, 14) 	Type':<class 'pandas.core.frame.DataFrame'>
Grupo:('Hombre', np.float64(8.0)), 	Tamaño:(15608, 14) 	Type':<class 'pandas.core.frame.DataFrame'>
Grupo:('Hombre', np.float64(9.0)), 	Tamaño:(10163, 14) 	Type':<class 'pandas.core.frame.DataFrame'>
Grupo:('Hombre', np.float64(10.0)), 	Tamaño:(10467, 14) 	Type':<class 'pandas.core.frame.DataFrame'>
Grupo:('Hombre', np.float64(11.0)), 	Tamaño:(13963, 14) 	Type':<class 'pandas.core.frame.DataFrame'>
Grupo:('Hombre', np.float64(12.0)), 	Tamaño:(22056, 14) 	Type':<class 'pandas.core.frame.DataFrame'>
G

/tmp/ipykernel_1810/4159702069.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = df.groupby(['sexo','mes_ocurr'])


Las funciones de agregación son funciones en Pandas que se utilizan en conjunto con la función `groupby()` para realizar cálculos resumidos sobre un conjunto de datos agrupados. Estas funciones toman un conjunto de valores y los resumen en un solo valor. Estas son las funciones que vimos anteriormente en las tablas pivote, las puedes consultar en [funciones de agrupamiento](https://pandas.pydata.org/pandas-docs/stable/reference/groupby.html#seriesgroupby-computations-descriptive-stats).

Por ejemplo, agrupemos los datos por sexo y área urbana/rural y calculemos el promedio de edad. Fijate *en* el orden de las selecciones y operaciones.

In [91]:
df.groupby(by=['sexo','mes_ocurr'])['edad'].mean()

/tmp/ipykernel_1810/2909050276.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(by=['sexo','mes_ocurr'])['edad'].mean()


sexo             mes_ocurr
Hombre           3.0          57.613333
                 4.0          57.598975
                 5.0          59.522409
                 6.0          61.213028
                 7.0          62.375881
                 8.0          63.027813
                 9.0          63.594685
                 10.0         63.696637
                 11.0         64.358269
                 12.0         63.605180
Mujer            3.0          64.080000
                 4.0          60.902830
                 5.0          62.139815
                 6.0          63.322964
                 7.0          64.014486
                 8.0          64.761097
                 9.0          65.342516
                 10.0         65.622348
                 11.0         65.789031
                 12.0         65.702180
No especificado  3.0                NaN
                 4.0                NaN
                 5.0           0.000000
                 6.0                NaN
                 7.0                NaN
                 8.0                NaN
                 9.0                NaN
                 10.0         76.000000
                 11.0               NaN
                 12.0               NaN
Name: edad, dtype: float64

La función `.aggregate()` o `.agg()` se utiliza para aplicar una o varias funciones de agregación a un DataFrame. Esta función es muy útil cuando queremos aplicar diferentes funciones de agregación a diferentes columnas de un DataFrame o cuando queremos aplicar funciones de agregación personalizadas.

También se pueden aplicar distintas funciones a cada columna usando el siguiente formato.

```
.agg(
     result_col1=(target_col1, function1),
     result_col2=(target_col2, function2),
     result_col3=(target_col2, function2),
)
```

Estas funciones pueden incluir funciones definidas en `pandas`, de otras bibliotecas o propias. En caso usar funciones con argumentos es necesario usar una función `lambda` envolviendo la función.

Por ejemplo, agrupemos por `mes_ocurr` y calculemos varias estadísticas de la `edad` y `sexo` para cada categoría de `sexo`.

In [92]:
df.groupby('mes_ocurr').agg(
    edad_promedio = ('edad','mean'),
    edad_min = ('edad','min'),
    edad_mediana = ('edad','median'),
    edad_max = ('edad','max'),
    sexo_moda = ('sexo', pd.Series.mode ),
    )

,edad_promedio,edad_min,edad_mediana,edad_max,sexo_moda
mes_ocurr,,,,,
3.0,59.230000,2.0,57.5,95.0,Hombre
4.0,58.635511,0.0,59.0,102.0,Hombre
5.0,60.373844,0.0,61.0,108.0,Hombre
6.0,61.930135,0.0,62.0,104.0,Hombre
7.0,62.966376,0.0,64.0,107.0,Hombre
8.0,63.643949,0.0,64.0,110.0,Hombre
9.0,64.245397,0.0,65.0,114.0,Hombre
10.0,64.411910,0.0,65.0,103.0,Hombre
11.0,64.892411,0.0,66.0,110.0,Hombre


## Ejercicio

Genera una tabla pivote con las dos variables que agregaste en la sección anterior con el tamaño de los grupos. Incluye los totales y porcentajes.

In [93]:
data = df.pivot_table(index='edad_class',columns='sexo', aggfunc='size')
data.loc['Total sexo'] = data.sum(axis=0)
data['Total edad'] = data.sum(axis=1)
data.loc['Total sexo','Total edad'] = 1
data

/tmp/ipykernel_1810/3236403046.py:1: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  data = df.pivot_table(index='edad_class',columns='sexo', aggfunc='size')


sexo,Hombre,Mujer,No especificado,Total edad
edad_class,,,,
Adulto,51517,23832,0,75349
Mayor,76895,47358,1,124254
Menor,327,264,1,592
Total sexo,128739,71454,2,1


## Resumen

En esta lección hemos aprendido varios conceptos:


* Funciones útiles: `.mean()`, `.median()`, `.std()`, `.sum()`, `.size()`, `.count()`, `.first()`, `.last()`, `.nth()`, `.min()`, `.max()`

* Tablas pivote: resumen y reorganizan datos
    * Índice: categoría principal
    * Columnas: categoría adicional
    * Valores: datos a resumir
    * Función: función para resumir los datos
    * Nota: se puede usar un multi-índice para incorporar variables

* Concat: unir tablas uno a uno dependiendo del eje
    * Una sobre otra `axis=0`
    * Lado a lado `axis=1`

* Merge: unir tablas muchos a uno usando los valores de las columnas
    * Determinar que columna(s) unir en cada tabla (izquierda y derecha)
    * Existen varias formas de unir tablas: `inner`, `outer`, `left`, `right`

* El patrón División-Aplicación-Combinación (split-apply-combine) permite realizar análisis por subgrupos de forma eficiente y flexible.
    1. Dividir el conjunto de datos en grupos con `groupby()`
    2. Aplicar funciones como agregación, transformación o filtrado
    3. Combinar los resultados en un nuevo DataFrame

* La función agregar `.agg()` permite hacer cálculos resumiendo cada grupo en un parámetro
    * Se pueden usar funciones predefinidas con cadenas de texto, de `pandas`, otras bibliotecas o `lambda`
    * Formato para múltiples columnas y funciones (recuerda que los nombres de columnas van sin comillas):
    ```
      .agg(
            result_col1=(target_col1, function1),
            ...,
          )
    ```

Estas son funciones muy poderosas, por lo que en este curso introductorio solo veremos las bases. Puedes aprender más en las guías de usuario y documentación de las funciones:
* [Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html)
* [Reshaping and pivot tables](https://pandas.pydata.org/docs/user_guide/reshaping.html)
* [Group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html)
* Documentación: [`pd.pivot_table()`](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html), [`pd.concat()`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html), [`pd.merge()`](https://pandas.pydata.org/docs/reference/api/pandas.merge.html), [`.groupby()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) y [`.aggregate()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.aggregate.html)

**¡Gracias!**